# Tutorial 3: Variable-length trial sequences (future support)

> **Status — partially supported / under active development.** The
> mask-based batching workflow described in this tutorial is *planned*;
> only the building blocks listed below are available today. Until equal-
> length padding and masking land in the public API, treat trials inside a
> single batch as if they must share a fixed `T` (see "Current limitation"
> below). Track the changelog for updates.

This tutorial covers:

1. What works today for trials whose phase structure varies.
2. Why `batch_sample` requires a fixed `T` across the batch.
3. The planned padding-plus-mask API (subject to change).
4. Workarounds you can use *today* to train on variable-length trials.
5. Extension points for users who want to experiment ahead of the
   official API.

---


In [1]:
import brainunit as u
import brainstate
import jax.numpy as jnp

brainstate.environ.set(dt=1.0 * u.ms)

from braintools.cogtask import (
    Task, Feature, concat,
    Fixation, Delay, Response, Stimulus, Blank, Sample,
    If, Switch, While,
    TruncExp, UniformDuration,
    von_mises,
)


## 1. What works today

### Per-trial scalar sampling

`TruncExp` and `UniformDuration` are callable distributions over
`brainunit` quantities. They are typically used inside `trial_init` to
draw a quantity that is then stored in `Context` for an encoder or
predicate to read.


In [2]:
delay_dist = TruncExp(mean=600 * u.ms, min_val=300 * u.ms, max_val=1500 * u.ms)

def trial_init(ctx):
    ctx['ground_truth'] = int(ctx.rng.choice(2))
    ctx['target_delay'] = delay_dist(ctx)   # JIT/vmap-safe (uses ctx.rng)

# Demonstrate sampling outside a task:
samples = [float(delay_dist().mantissa) for _ in range(5)]
samples


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


[412.5071105957031,
 801.7503051757812,
 680.4615478515625,
 938.8588256835938,
 631.2080078125]

### Conditional and looped phases

`If`, `Switch`, and `While` already support data-dependent control flow
inside a single trial. The *upper-bound* duration determines the trial's
tensor size: `While.get_duration` multiplies `body`'s duration by
`max_iterations`, and `If` / `Switch` resolve their branch during the
dry-run pass.


In [3]:
loop = While(
    condition=lambda ctx: ctx.get('evidence', 0.0) < ctx['threshold'],
    body=Sample(50 * u.ms,
                inputs={'stimulus': lambda ctx, f: ctx['pulse']},
                outputs={'label': 0}),
    max_iterations=20,
)
# Worst-case duration is body * max_iterations regardless of how early the
# loop actually exits.
loop


While(body=Sample, max=20)

### Single-trial JIT

`task.sample(index)` is JIT-compiled per trial, so a variable-duration
trial works in eager-execution training loops where each trial is
processed independently. The cost is that you lose `vmap`-level
parallelism — see the limitation below.

---


## 2. Current limitation

`task.batch_sample` uses `vmap` over the trial index, which requires that
every trial in the batch produce buffers of **identical shape**.
Concretely, this means:

- The same root key plus the same dry-run pass must yield the same total
  duration `T` across all batch elements.
- If `trial_init` samples a duration that differs across trials (e.g. via
  `TruncExp`), `batch_sample` cannot stack the results.

Variable lengths *across the batch axis* are not yet expressible in the
JIT/`vmap` path. They work in plain Python loops over `sample_trial`, but
you lose the speedup.

---


## 3. Planned design: padding + masks

The intended extension makes `batch_sample` JIT- and `vmap`-compatible
for variable-length trials by allocating buffers at a fixed `T_max` and
exposing a per-trial mask:

1. **Static upper bound.** Tasks declare `T_max` (either explicitly or
   inferred from the worst-case phase durations).
2. **Padded buffers.** Each trial allocates `(T_max, num_inputs)` and
   `(T_max,)` or `(T_max, num_outputs)` regardless of its actual length.
3. **Mask channel.** `batch_sample` returns an additional `mask` of shape
   `(T_max, B)` (or `(B, T_max)`) with `True` over the valid region and
   `False` over padding.
4. **Encoder contract.** Declarative phases already write only into their
   own slice `[phase_start:phase_end]`, so the padded region remains at
   the default value (`0` for inputs and labels).
5. **Loss helpers.** A `masked_loss` utility will reduce loss and metrics
   over the valid steps only.

Proposed API sketch *(NOT YET IMPLEMENTED — illustrative only)*:

```python
task = MyVariableLengthTask(t_max=4000 * u.ms, seed=0)
X, Y, mask = task.batch_sample(64, return_mask=True)
# X:    (T_max, 64, num_inputs)
# Y:    (T_max, 64)
# mask: (T_max, 64)  bool — True where the trial is "live"

from braintools.cogtask import masked_cross_entropy   # planned helper
loss = masked_cross_entropy(logits, Y, mask)
```

---


## 4. Workarounds available today

### 4a. Cap to a fixed maximum and write only into the live region

Keep every phase's wall-clock duration fixed at the worst case, and use a
per-trial *active length* stored in `Context` to gate the encoders.
Inputs and labels outside the live region remain at the default value
(zeros), and your loss function can read the active length from the
metadata returned by `batch_sample(..., return_meta=True)`.


In [4]:
MAX_DELAY = 1500 * u.ms

class GatedDelayTask(Task):
    def define_features(self):
        fix    = Feature(1, 'fixation')
        stim   = Feature(2, 'stimulus')
        choice = Feature(2, 'choice')
        return fix + stim, fix + choice

    def define_phases(self):
        def gated_fixation(ctx, feat):
            T = ctx.phase_duration
            live = jnp.asarray(ctx['delay_steps'], dtype=jnp.int32)
            mask = (jnp.arange(T) < live).astype(jnp.float32)
            return jnp.broadcast_to(mask[:, None], (T, feat.num))

        return concat([
            Fixation(100 * u.ms, inputs={'fixation': 1.0},  outputs={'label': 0}),
            Delay(MAX_DELAY,     inputs={'fixation': gated_fixation},
                                 outputs={'label': 0}),                   # fixed shape, variable signal
            Response(100 * u.ms, outputs={'label': lambda c, f: c['choice'] + 1}),
        ])

    def trial_init(self, ctx):
        # Sample the *active* delay length once per trial (a JAX scalar).
        ctx['delay_steps'] = ctx.rng.randint(300, 1500)
        ctx['choice']      = ctx.rng.choice(2)

    def get_trial_meta(self, trial_state):
        return {'delay_steps': trial_state['delay_steps']}

task = GatedDelayTask(seed=0)
X, Y, meta = task.batch_sample(8, return_meta=True)
print('X.shape =', X.shape, 'Y.shape =', Y.shape)
print('per-trial active delay_steps =', meta['delay_steps'])


X.shape = (1700, 8, 3) Y.shape = (1700, 8)
per-trial active delay_steps = [ 368 1372  342  960 1399 1034  821  844]


### 4b. Bucketed batching

Group trials by length *before* calling `batch_sample` so each batch is
internally uniform. The cost is reduced randomness in batch composition;
the benefit is that you get full JIT/`vmap` throughput.

### 4c. Explicit mask channel as an input feature

Add an extra input feature that is `1.0` during live timesteps and `0.0`
during padding. This lets the model learn to ignore padding even before
the framework provides built-in masks.

```python
mask_in = Feature(1, 'mask')
inputs  = fix + stim + mask_in
```

Inside each phase, set `'mask': 1.0` during the live region and `'mask':
0.0` (or omit it — the default is zero) in a trailing `Blank` padding
phase.

---


## 5. Extension points

If you need variable-length support before the official API lands, the
cleanest extension points are:

- **Subclass `Task`** and override `Task.sample_trial` to allocate
  buffers at a fixed `T_max` and emit a mask. The phase tree itself does
  not need to change — phases already write only into
  `[phase_start:phase_end]`.
- **Tighten upper bounds on conditional phases.** `While.get_duration`
  already uses `max_iterations`; similar reasoning lets you wrap `If` and
  `Switch` so `get_duration` returns the max over branches.
- **Implement a custom compound phase** (subclass `Phase` with
  `IS_COMPOUND = True`) that owns its own padded sub-buffer and writes a
  mask channel into the parent context.

Track the project changelog for the official mask-based API; this
tutorial will be updated when it is part of the public surface.
